# How many bikes go out tomorrow?

You have taken a job with the bike hire operator in Washington DC.

Every night a crew checks, cleans and redistributes bikes around the city.
The crew is booked the evening before. Book too few and broken bikes sit at
empty stations all day. Book too many and you have paid people to stand
about.

Your manager asks you one question, and she wants a number rather than a
chart:

**Given tomorrow's weather forecast, how many hires should we expect?**

She is going to staff against your answer, so she also wants to know how far
out that number is likely to be.

This notebook walks the whole job end to end: get the data into the right
shape, hold some back, set a bar, fit a model, read what it says, make the
prediction, and then find out - the hard way - that the way you split your
data decides whether you believe a model that does not work.

*Data: UCI Bike Sharing Dataset, hourly hire counts for Capital Bikeshare in
Washington DC, 2011 and 2012, assembled by Hadi Fanaee-T and João Gama. The
same file is used in the Module 2 notebooks, so some of it will look
familiar.*

## How to work through this

Every step asks a question before it answers it. **Commit to an answer
before you run the cell** - out loud, in the chat, or on paper. A wrong
guess costs nothing and is worth more than a right one, because the
surprise is what you remember afterwards.

Nothing here needs anything installed beyond what the course environment
already has: `pandas`, `matplotlib` and `scikit-learn`.

## Step 1. Get the data, and notice it is the wrong shape

The file records **one row per hour**. Your manager asks about **days**.

That mismatch is the first real task, and it is the kind of thing that never
appears in a tidy tutorial: the data you are given is almost never in the
shape the question needs.

Before I run the next cell: two years of hourly records, one row per hour.
Roughly how many rows would you expect?

In [ ]:
import os

import pandas as pd

# The file lives in data/ - either the cohort folder's copy or the course
# repository's. Try both names so this notebook runs in either place.
def find_data():
    """Return the first path that exists, or stop with a clear message."""
    candidates = [
        os.path.join("data", "bikeshare-hour.csv"),   # cohort folder
        os.path.join("data", "bikeshare.csv"),        # course repository
        os.path.join("..", "data", "bikeshare.csv"),        # a module folder
        os.path.join("..", "..", "..", "data", "bikeshare.csv"),  # repo root
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise SystemExit("bikeshare hourly file not found. Looked in: %s" % candidates)

HOURLY_FILE = find_data()
print("reading:", HOURLY_FILE)

# parse_dates turns the date column from text into real dates, so that
# pandas can sort by it and pull the month or the year out later.
hourly = pd.read_csv(HOURLY_FILE, parse_dates=["dteday"])

print("rows:", len(hourly))
print("columns:", list(hourly.columns))
hourly.head()

### What the columns mean

The ones this notebook uses:

| Column | What it holds |
| :--- | :--- |
| `dteday` | the date |
| `hr` | hour of the day, 0 to 23 |
| `cnt` | hires in that hour - **this is what we want to predict** |
| `casual`, `registered` | the two kinds of rider. They add up to `cnt` |
| `temp` | temperature, but **not in degrees** - see step 3 |
| `hum` | humidity, also rescaled |
| `windspeed` | wind, also rescaled |
| `workingday` | 1 on a working weekday, 0 at weekends and on holidays |
| `weathersit` | a weather code, 1 clear to 4 severe |

`cnt` is the **target**: the thing we want. Everything else we might use to
predict it is a **feature**. That vocabulary is from `17-regression-basics`
and it does not change: features in, target out.

## Step 2. Turn hours into days

To go from hourly rows to daily rows we **group** all the rows that share a
date and collapse each group into one row.

The part worth thinking about is that not every column collapses the same
way:

- **Hires add up.** Twenty-four hourly counts become one daily total.
- **Weather averages.** A day does not have one temperature, so we take the
  mean of its hours.
- **Working day does not do either.** Every hour of one day carries the same
  0 or 1, so any single value will do. We take the maximum, which just picks
  that shared value.

Before I run it: how many rows should the daily table have, for the whole of
2011 and 2012?

In [ ]:
# groupby collects the rows that share a date. .agg then says what to do
# with each column of each group: sum the rides, average the weather.
daily = hourly.groupby("dteday").agg(
    rides=("cnt", "sum"),
    casual=("casual", "sum"),
    registered=("registered", "sum"),
    temp=("temp", "mean"),
    humidity=("hum", "mean"),
    wind=("windspeed", "mean"),
    workingday=("workingday", "max"),
    weather_code=("weathersit", "max"),
).reset_index()

print("daily rows:", len(daily))
print("first day :", daily["dteday"].min().date())
print("last day  :", daily["dteday"].max().date())
daily.head()

### Why that number and not 730

365 plus 365 is 730. The table has one more, because **2012 was a leap
year** and had a 29th of February.

This is worth a moment. If you had assumed 730 and gone looking for the
"extra" row as an error, you would have deleted a real day. Counting rows
and being able to explain the count is not box-ticking - it is how you catch
the difference between data that is wrong and data that is merely
surprising.

Now a second check. We claimed the daily total is the sum of the hourly
counts. Rather than trust that, we test it: rebuild the total a second way
and demand the two agree.

In [ ]:
# A sanity check: the rides column should equal the sum of casual and
# registered, for every single day. If that is ever false, the aggregation
# above is wrong and everything after it is built on sand.
rebuilt = daily["casual"] + daily["registered"]
disagreements = (rebuilt != daily["rides"]).sum()

print("days where casual + registered does not equal rides:", disagreements)
assert disagreements == 0, "the two ways of counting disagree"
print("they agree on all", len(daily), "days.")

# A day has 24 hours. Did any day arrive with fewer rows than that?
hours_per_day = hourly.groupby("dteday").size()
print()
print("days with fewer than 24 hourly rows:", (hours_per_day < 24).sum())
print("the fewest hours any day has:", hours_per_day.min())

Some days have fewer than 24 rows. The hours with no hires at all were never
recorded, rather than recorded as zero.

For a daily total this does not matter - adding nothing adds nothing. But
notice what it would do to an **average**: the mean of the hours that exist
is not the mean of the hours in the day. Our weather averages are therefore
slightly weighted towards the hours when someone was riding.

That is a real limitation of this model, and the right thing to do with it
is write it down rather than pretend it is not there.

## Step 3. Put the weather back into units a human can read

The three weather columns are not in real units. Whoever assembled this file
squashed each one onto a scale from 0 to 1 by dividing by its maximum. The
documentation gives the divisors:

$$
\text{temperature in } ^\circ C = \texttt{temp} \times 41
$$

$$
\text{humidity in } \% = \texttt{humidity} \times 100
$$

$$
\text{wind in km/h} = \texttt{wind} \times 67
$$

You could skip this step and the model would fit exactly as well. The
predictions would be identical. **What you would lose is the ability to read
the model.** A coefficient of 160 means something when the column is degrees
Celsius. When the column is a squashed number between 0 and 1, a coefficient
of 6570 means nothing to the manager and nothing to you.

Before I run it: on that 0-to-1 scale, what do you think the hottest day in
the file comes out as in degrees?

In [ ]:
# The divisors come from the dataset documentation, not from us.
daily["temp_c"] = daily["temp"] * 41
daily["humidity_pct"] = daily["humidity"] * 100
daily["wind_kph"] = daily["wind"] * 67

print("temperature : %.1f to %.1f degrees C" % (daily["temp_c"].min(), daily["temp_c"].max()))
print("humidity    : %.0f to %.0f %%" % (daily["humidity_pct"].min(), daily["humidity_pct"].max()))
print("wind        : %.1f to %.1f km/h" % (daily["wind_kph"].min(), daily["wind_kph"].max()))
print()
print("rides in a day: %d to %d, average %.0f"
      % (daily["rides"].min(), daily["rides"].max(), daily["rides"].mean()))

## Step 4. Look at it before you model it

A model cannot tell you whether a relationship exists. It can only tell you
how strong the straight-line version of it is, and it will happily report a
number when the answer is "there is nothing here".

So we look first.

Before I run it: sketch the shape you expect for daily hires against daily
temperature. A straight rise? A curve? A cloud with no pattern?

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(7, 5))
axis.scatter(daily["temp_c"], daily["rides"], s=12, alpha=0.5)
axis.set_xlabel("average temperature for the day (degrees C)")
axis.set_ylabel("hires that day")
# The title stays neutral on purpose - the next cell asks you to commit to
# a number, and a title naming the finding would answer it for you.
axis.set_title("Every day in 2011 and 2012: temperature across, hires up")
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

Two things are visible that the model will not tell you:

The cloud **rises**, so warmer days do bring more riders. But it also
**bends over and comes back down** at the warm end - the hottest days are
not the busiest. A straight line cannot say that. It will be pulled through
the middle and will be wrong at both extremes in the same direction.

And there is a **band of very low days scattered all the way along**,
including at pleasant temperatures. Something other than temperature ruined
those days.

Let us find the worst one.

In [ ]:
# .idxmin() gives the row label of the smallest value; .loc fetches that row.
quietest = daily.loc[daily["rides"].idxmin()]
busiest = daily.loc[daily["rides"].idxmax()]

print("quietest day: %s  %5d hires  %.1f C  weather code %d"
      % (quietest["dteday"].date(), quietest["rides"],
         quietest["temp_c"], quietest["weather_code"]))
print("busiest day : %s  %5d hires  %.1f C  weather code %d"
      % (busiest["dteday"].date(), busiest["rides"],
         busiest["temp_c"], busiest["weather_code"]))

In [ ]:
daily

The quietest day is not cold. It is mild.

**Look up what happened in Washington DC on that date.** The answer is not
in this file, and no amount of modelling will recover it. Knowing when to
stop staring at a table and go and read something is part of the job.

Keep that day in mind. When the model is wrong later, it will be wrong about
days like this one.

## Step 5. Hold some data back before you fit anything

Here is the trap the whole of Module 4 is built around.

If you fit a model on every row you have, and then measure how well it does
**on those same rows**, you learn almost nothing. A model can score well by
memorising rather than by learning, and memorising looks identical to
learning until something new arrives.

So we hide some data. The model never sees it while it learns. Then we test
on the hidden part, which is the closest we can get to asking "what would
this do tomorrow?" without waiting for tomorrow.

The usual split is about 80 per cent to learn from, 20 per cent held back.
`random_state=42` fixes which rows get chosen, so that everyone in the room
gets the same split and the same numbers. Any fixed value works; the point
is that it is fixed.

In [ ]:
from sklearn.model_selection import train_test_split

FEATURES = ["temp_c", "humidity_pct", "wind_kph", "workingday"]
TARGET = "rides"

X = daily[FEATURES]
y = daily[TARGET]

# test_size=0.2 holds back a fifth. random_state fixes the choice so the
# result is the same every time this runs.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("days to learn from :", len(X_train))
print("days held back     :", len(X_test))
print("total              :", len(X_train) + len(X_test))

In [ ]:
X_train

## Step 6. Set the bar before you try to clear it

A number like "my model is out by 1400 hires" means nothing on its own. Out
by 1400 compared with **what**?

So before fitting anything, we work out how wrong the laziest possible
answer is. That answer ignores the weather, ignores the day of the week, and
predicts the same number every day: the average of the days it learned from.

Any model that cannot beat this is not earning its keep.

We score it two ways, because they say different things:

$$ \text{MAE} = \frac{1}{n}\sum_{i=1}^{n} \left| y_i - \hat{y}_i \right| $$

$$ \text{RMSE} = \sqrt{ \frac{1}{n}\sum_{i=1}^{n} \left( y_i - \hat{y}_i \right)^2 } $$

MAE is the average miss, in hires. RMSE squares each miss before averaging,
which makes big misses hurt far more than small ones, then takes the square
root to get back to hires. Both are in the manager's units. Use MAE to say
"typically out by this much", RMSE when a single catastrophic day matters
more than several small errors.

Before I run it: the busiest day had over eight thousand hires and the
quietest had almost none. If we predict the average every single day, how
far out do you think we will typically be?

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# The laziest model: one number, said every day, learned from the training
# days only. It must never look at the held-back days, not even to average
# them - that would be letting it see the answers.
always_predict = y_train.mean()
print("the one number this model always says: %.0f hires" % always_predict)

baseline_prediction = [always_predict] * len(y_test)

baseline_rmse = root_mean_squared_error(y_test, baseline_prediction)
baseline_mae = mean_absolute_error(y_test, baseline_prediction)

print()
print("on the held-back days:")
print("  MAE  : %.0f hires" % baseline_mae)
print("  RMSE : %.0f hires" % baseline_rmse)
print()
print("That is the bar. Everything from here has to beat it.")

## Step 7. One feature: does temperature alone beat the bar?

Now the model. `LinearRegression` finds the straight line

$$ \hat{y} = \beta_0 + \beta_1 x $$

that makes the squared misses as small as they can collectively be. That is
the same machinery as `17-regression-basics` - nothing new is happening, the
columns have just changed.

Three lines do the whole thing:

- `LinearRegression()` makes an empty model that knows nothing
- `.fit(X_train, y_train)` shows it the training days and works out the two numbers
- `.predict(X_test)` applies them to days it has never seen

Before I run it: temperature alone, against a model that ignores everything.
How much of the bar do you think it takes off?

In [ ]:
from sklearn.linear_model import LinearRegression

temp_model = LinearRegression()
temp_model.fit(X_train[["temp_c"]], y_train)

temp_prediction = temp_model.predict(X_test[["temp_c"]])

temp_rmse = root_mean_squared_error(y_test, temp_prediction)
temp_mae = mean_absolute_error(y_test, temp_prediction)

print("what the model learned")
print("  intercept : %.0f hires" % temp_model.intercept_)
print("  slope     : %.1f hires per degree C" % temp_model.coef_[0])
print()
print("on the held-back days")
print("  baseline RMSE      : %.0f" % baseline_rmse)
print("  temperature RMSE   : %.0f" % temp_rmse)
print("  baseline MAE       : %.0f" % baseline_mae)
print("  temperature MAE    : %.0f" % temp_mae)
print()
print("error removed by knowing the temperature: %.0f hires (%.0f%%)"
      % (baseline_rmse - temp_rmse,
         100 * (baseline_rmse - temp_rmse) / baseline_rmse))

## Step 8. Four features

Temperature is not the only thing that keeps people off a bike. Add
humidity, wind, and whether it was a working day.

Nothing about the code changes except which columns go in. That is the
point of the matrix form in `17-regression-basics`: one column or four, it
is the same single multiplication, and scikit-learn does not care.

Before I run it: three more columns. Do they buy as much as temperature did
on its own?

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

prediction = model.predict(X_test)

rmse = root_mean_squared_error(y_test, prediction)
mae = mean_absolute_error(y_test, prediction)

print("RMSE in hires, on the held-back days")
print("  predict the average : %.0f" % baseline_rmse)
print("  temperature only    : %.0f" % temp_rmse)
print("  all four features   : %.0f" % rmse)
print()
print("MAE in hires")
print("  predict the average : %.0f" % baseline_mae)
print("  temperature only    : %.0f" % temp_mae)
print("  all four features   : %.0f" % mae)

In [ ]:
model.intercept_

In [ ]:
model.coef_

## Step 9. Say what the model learned, in English

A coefficient is a sentence waiting to be written out. Each one says: *hold
everything else still, move this one column up by one of its units, and the
prediction moves by this much.*

Because we put the weather back into real units in step 3, each of these
sentences is readable. Read every one aloud. If a sign surprises you, say so
rather than nodding it through - a coefficient with the wrong sign is one of
the few warnings a linear model gives you for free.

Before I run it: which of the four do you expect to matter most per unit,
and which sign will each one have?

In [ ]:
UNITS = {
    "temp_c": "degree C warmer",
    "humidity_pct": "percentage point more humid",
    "wind_kph": "km/h more wind",
    "workingday": "step from a weekend to a working day",
}

print("starting point (intercept): %.0f hires" % model.intercept_)
print()
for position in range(len(FEATURES)):
    name = FEATURES[position]
    coefficient = model.coef_[position]
    direction = "more" if coefficient >= 0 else "fewer"
    print("%-14s %+8.1f  ->  each %s brings about %.0f %s hires"
          % (name, coefficient, UNITS[name], abs(coefficient), direction))

The working-day coefficient deserves a second look, and it is a good example
of why reading coefficients beats trusting them.

It compares a working day with a weekend **while holding the weather still**.
It does not say weekends are quiet - it says that once you know the weather,
knowing whether people are at work adds little. Commuters ride on working
days; everyone else rides at weekends; the two largely cancel.

The Module 4 Part 2 notebook picks this up as *feature selection*: a column
can be genuinely related to the answer and still add nothing once the other
columns are present.

## Step 10. $R^2$, and what it actually means

RMSE and MAE are in hires, which is what the manager wants. $R^2$ is a
different question, with no units at all:

$$ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $$

The bottom is how much the answers vary around their own average - all the
variation there is to explain. The top is how much is left over after the
model has done its best. So $R^2$ is **the share of the variation the model
accounts for**. One would be perfect, zero means no better than predicting
the average.

We compute it twice - once from the formula, once with scikit-learn - and
demand they agree, so the definition is something you have watched happen
rather than something you have been told.

In [ ]:
from sklearn.metrics import r2_score

# Straight from the formula, with loops, so every part is visible.
answers = y_test.tolist()
predictions = prediction.tolist()
average_answer = sum(answers) / len(answers)

left_over = 0.0
total_variation = 0.0
for position in range(len(answers)):
    miss = answers[position] - predictions[position]
    spread = answers[position] - average_answer
    left_over = left_over + miss * miss
    total_variation = total_variation + spread * spread

r_squared_by_hand = 1 - left_over / total_variation

print("by hand      : %.4f" % r_squared_by_hand)
print("scikit-learn : %.4f" % r2_score(y_test, prediction))
assert abs(r_squared_by_hand - r2_score(y_test, prediction)) < 1e-9
print("they agree.")
print()
print("In words: the model accounts for %.0f%% of the day-to-day variation"
      % (100 * r_squared_by_hand))
print("in hires. The rest is things it cannot see.")

## Step 11. Answer the manager's question

Three forecasts land on your desk:

| | temperature | humidity | wind | working day? |
| :--- | ---: | ---: | ---: | :--- |
| A | 25 °C | 50 % | 12 km/h | yes |
| B | 8 °C | 80 % | 25 km/h | yes |
| C | 30 °C | 45 % | 10 km/h | no, a Sunday |

She gets a number for each. She also gets a range, because a single number
pretends to a precision the model does not have. RMSE is the natural width:
roughly, most days land within one RMSE of the prediction.

Before I run it: put your own number on day A.

In [ ]:
# One row per forecast, columns in the same order the model was fitted with.
forecasts = pd.DataFrame(
    [[25, 50, 12, 1],
     [8, 80, 25, 1],
     [30, 45, 10, 0]],
    columns=FEATURES,
    index=["A", "B", "C"],
)

forecast_prediction = model.predict(forecasts)

print("what to tell the manager")
print()
for position in range(len(forecasts)):
    label = forecasts.index[position]
    expected = forecast_prediction[position]
    print("  day %s: about %.0f hires, most likely between %.0f and %.0f"
          % (label, expected, expected - rmse, expected + rmse))

In [ ]:
forecasts

### One of those ranges is impossible

Look at the bottom of day B's range. It is **negative**, and a negative
number of bike hires cannot happen.

Nothing is broken. A straight line has no idea that hires stop at zero, and
"prediction plus or minus one RMSE" is a rough band rather than a real
probability. On a quiet, cold, wet day the band runs off the end of what is
physically possible.

What to do about it is a judgement call, not a calculation. You could report
the range as "0 to 2745" and say why. You could say the model is least
reliable on the quietest days. What you must not do is hand the manager a
negative number with a straight face because the code produced it.

Now the part that separates a person who can run scikit-learn from a person
who can be trusted with the answer.

**Which of those three would you trust least?**

Day C sits at 30 °C. Look again at the plot in step 4: the cloud bends down
at the warm end, and a straight line cannot bend. At 30 degrees the line is
still climbing while the real days have started to fall. The model will
overstate day C, and it will do so confidently.

Nothing in the RMSE tells you that. The scatter plot did.

## Step 12. Split it the way life does

Everything so far used a **random** split: a fifth of the days picked out at
random from across both years.

Think about what that means. The model learned from days in July 2012 and
was tested on days in June 2012. It had already seen the neighbours of every
day it was tested on.

Real life never works like that. Tomorrow is always after everything you
have. So we split by **time** instead: learn from the earlier days, test on
the later ones, exactly as you would have had to do on the day.

Same features. Same model. Same code. Only the split changes.

Before I run it: how different can the numbers be? The data is identical.

In [ ]:
# Sort by date, then cut at 80 per cent through the calendar rather than at
# random.
ordered = daily.sort_values("dteday").reset_index(drop=True)
cut_position = int(len(ordered) * 0.8)

earlier = ordered.iloc[:cut_position]
later = ordered.iloc[cut_position:]

print("learn from : %s to %s  (%d days)"
      % (earlier["dteday"].min().date(), earlier["dteday"].max().date(), len(earlier)))
print("test on    : %s to %s  (%d days)"
      % (later["dteday"].min().date(), later["dteday"].max().date(), len(later)))

time_model = LinearRegression()
time_model.fit(earlier[FEATURES], earlier[TARGET])
time_prediction = time_model.predict(later[FEATURES])

time_rmse = root_mean_squared_error(later[TARGET], time_prediction)
time_mae = mean_absolute_error(later[TARGET], time_prediction)
time_r2 = r2_score(later[TARGET], time_prediction)

print()
print("                     random split    time split")
print("RMSE (hires)        %10.0f    %10.0f" % (rmse, time_rmse))
print("MAE  (hires)        %10.0f    %10.0f" % (mae, time_mae))
print("R squared           %10.3f    %10.3f" % (r2_score(y_test, prediction), time_r2))

### Stop here and read that table again

The $R^2$ has gone **negative**.

That is not a bug and it is not an impossible number. Look at the formula in
step 10: $R^2$ is one minus a ratio, and nothing stops that ratio exceeding
one. A negative $R^2$ means the model does **worse than predicting the
average every day**. You would have been better off with the lazy model from
step 6.

The same model, on the same data, with the same code. Only the split
changed.

Before the next cell: is the model wrong about the weather, or wrong about
something else? Look at the direction of its misses.

In [ ]:
misses = later[TARGET].values - time_prediction

print("average miss on the test period: %+.0f hires" % misses.mean())
print("days predicted too low : %d" % (misses > 0).sum())
print("days predicted too high: %d" % (misses < 0).sum())
print()
print("This is not random scatter. The model is short almost every day.")

A model that is wrong in **both** directions is noisy. A model that is wrong
in the **same** direction almost every day is missing something systematic -
and systematic is a much more serious word, because no amount of extra data
of the same kind will fix it.

The next plot shows what it is missing. Before I run it: what would make
every single prediction in the later period too low?

In [ ]:
figure, axis = plt.subplots(figsize=(12, 4.5))
axis.plot(ordered["dteday"], ordered["rides"], linewidth=0.8)
axis.axvline(ordered["dteday"].iloc[cut_position], color="red", linestyle="--")
axis.set_xlabel("date (the red line is where we cut)")
axis.set_ylabel("hires that day")
axis.set_title("Two years of daily hires, with the train and test cut marked")
axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()

first_year = ordered[ordered["dteday"].dt.year == 2011]["rides"].mean()
second_year = ordered[ordered["dteday"].dt.year == 2012]["rides"].mean()

print("average hires per day in 2011: %.0f" % first_year)
print("average hires per day in 2012: %.0f" % second_year)
print("growth: %+.0f%%" % (100 * (second_year - first_year) / first_year))

### The business grew

The seasonal wave is obvious - up in summer, down in winter, twice. But the
second wave sits **higher than the first**. More people used the service in
2012 than in 2011.

The model learned the level of the earlier period and kept predicting it.
It was never wrong about weather. It was blind to growth, because we never
gave it a column that could express "later".

And now the uncomfortable part: **the random split hid this completely.**
It sprinkled 2012 days through the training set, so the model had already
seen the higher level. The random split reported a healthy model. The time
split reported the truth about what would have happened in production.

Whenever your rows have an order - dates, versions, patients arriving - the
random split is the one that flatters you.

## Step 13. Give it a way to say "later"

One column: how many days have passed since the first day in the data.

It is a crude way to describe growth - it can only say "up by a constant
amount per day" - but it is enough to test whether growth really was the
problem.

Before I run it: one extra column, on the split that just produced a
negative score. How much of that can a single column recover?

In [ ]:
# Days since the start of the record. The model can now learn a trend.
ordered["day_number"] = (ordered["dteday"] - ordered["dteday"].min()).dt.days

earlier = ordered.iloc[:cut_position]
later = ordered.iloc[cut_position:]

WITH_TREND = FEATURES + ["day_number"]

trend_model = LinearRegression()
trend_model.fit(earlier[WITH_TREND], earlier[TARGET])
trend_prediction = trend_model.predict(later[WITH_TREND])

trend_rmse = root_mean_squared_error(later[TARGET], trend_prediction)
trend_r2 = r2_score(later[TARGET], trend_prediction)
trend_bias = (later[TARGET].values - trend_prediction).mean()

print("on the time split, testing on days the model has never seen")
print()
print("                        RMSE      R squared     average miss")
print("four features       %8.0f     %10.3f     %+12.0f" % (time_rmse, time_r2, misses.mean()))
print("plus a trend column %8.0f     %10.3f     %+12.0f" % (trend_rmse, trend_r2, trend_bias))
print()
print("trend learned: %+.1f extra hires per day that passes"
      % trend_model.coef_[len(FEATURES)])

One column, and the model goes from worse-than-useless to genuinely useful
on data it has never seen. The average miss collapses towards zero, which is
the sign that the systematic problem has gone and only noise is left.

Be careful what you conclude from it, though. A straight trend says growth
continues forever at the same rate. Push this model three years out and it
will predict numbers the city does not have bikes for. **A model is only
trustworthy inside the range of the data it learned from**, and that applies
to time as much as to temperature.

## What you should be able to do now

- Take data in the wrong shape and group it into the shape the question needs
- Check a count and explain it, rather than assuming it
- Put measurements back into units a human can read, so the model can be read
- Hold data back before fitting anything, and say why
- Set a baseline and state a model's worth as the distance from it
- Report RMSE and MAE in the units of the business question
- Compute $R^2$ from its formula and explain a negative one
- Turn coefficients into sentences, with units and signs
- Recognise when a random split is flattering you, and split by time instead
- Tell the difference between a model that is noisy and one that is biased

## If you want to take it further

1. Add the weather code as a feature. It is 1 to 4, and the model will treat
   it as a number - as though a code 4 day were four times a code 1 day. Is
   that a fair thing to assume? What would you do instead?
2. Fit `casual` and `registered` separately. Which is easier to predict, and
   does the reason make sense to you?
3. The straight line bends the wrong way at high temperatures. Add a
   temperature-squared column and see whether the warm end improves.
   `18-model-quality` is where this idea turns into a whole topic.
4. Your manager says being short is three times worse than being over.
   Nothing we have done knows that. What would you change, and what would
   you hand her?
5. Rerun the random split with `random_state=1`, then `2`, then `3`. How
   much do the numbers move? What does that tell you about quoting an RMSE
   to three significant figures?